In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("C:\\Users\\User\\Downloads\\MAYOR Y BALANCE JUNIO 25.xlsx", sheet_name="Report")

df.head()

In [ ]:
df = df.astype({
    "CP SUPREMO SAS": "string",
    "Unnamed: 1": "string",
    "Unnamed: 2": "string",
    "MAYOR Y BALANCES CON TERCEROS": "string",
    "Unnamed: 11": "string"
})


In [ ]:
df = df.rename(columns={
    "CP SUPREMO SAS": " Cuenta",
    "Unnamed: 1": "Column2",
    "Unnamed: 2": " Descripcion",
    "Unnamed: 3": "Column4",
    "Unnamed: 4": "Column5",
    "Unnamed: 5": "Column6",
    "Unnamed: 6": "Column7",
    "MAYOR Y BALANCES CON TERCEROS": "Column8",
    "Unnamed: 8": "Saldo anterior",
    "Unnamed: 9": "Column10",
    "Unnamed: 10": "Debitos ",
    "Unnamed: 11": "Column12",
    "Unnamed: 12": "Creditos ",
    "Unnamed: 13": "Column14",
    "Unnamed: 14": "Column15",
    "Unnamed: 15": "Saldo",
})

In [ ]:
df = df.iloc[6:].reset_index(drop=True)

In [ ]:
df = df.dropna(how="all")

In [ ]:
df = df.rename(columns={
    " Cuenta": "PUC",
    "Saldo anterior": "Saldo inicial",
    "Saldo": "Saldo final"
})

In [ ]:
df = df[["PUC","Column2"," Descripcion","Saldo inicial","Debitos ","Creditos ","Saldo final"]]

In [ ]:

# df["PUC"] = pd.to_numeric(df["PUC"], errors="coerce")
# df = df.dropna(subset=["PUC"])

original_nulls = df["PUC"].isna()

df["PUC"] = pd.to_numeric(df["PUC"], errors='coerce')

df_limpio = df[~df["PUC"].isna() | original_nulls].copy()

df.head(205)

In [ ]:
df = df.dropna(subset=["Saldo inicial"]) 

In [ ]:
df["Personalizado"] = np.where(df["Saldo inicial"].isna(), df[" Descripcion"], "")

df.head(25)

In [ ]:
df["Personalizado"] = df["Personalizado"].shift(-1)

In [ ]:
df["Combinada"] = (df[" Descripcion"].fillna("") + " " + df["Personalizado"].fillna("")).str.strip()

In [ ]:
df = df[df["Saldo inicial"].notna()]

In [ ]:
df = df.drop(columns=[
    "Column4","Column5","Column6","Column7","Column8",
    "Column10","Column12","Column14","Column15","Column17"
], errors="ignore")

In [ ]:
df["Col2_original"] = df["Column2"]

In [ ]:
df["Combinada"] = df["Combinada"].str.title()

In [ ]:
df["Combinada"] = df["Combinada"].str.title()

In [ ]:
df.head(25)

In [ ]:
df["Tercero"] = np.where(df["Col2_original"].notna(), df["Combinada"], "")

In [ ]:
df.head(25)

In [ ]:
df = df.drop(columns=["Column2"])
df["PUC"] = df["PUC"].astype(str)

In [ ]:
df["Personalizado"] = (df["PUC"] + "01").replace("nan01", np.nan)
df["Personalizado"] = df["Personalizado"].ffill()   # sin warning

In [ ]:
df["Personalizado.1"] = np.where(df["Tercero"]=="", df["PUC"], df["Personalizado"])

In [ ]:
df.head(25)

In [ ]:
df = df.drop(columns=["PUC","Personalizado"])
df = df.rename(columns={"Personalizado.1":"PUC","Combinada":"Descripción"})

In [ ]:
df["Cuenta"] = df["PUC"].str[0].map({
    "1":"Activo","2":"Pasivo","3":"Patrimonio",
    "4":"Ingreso","5":"Gasto"
}).fillna("Costo")

In [ ]:
df["Largo"] = df["PUC"].str.len()

In [ ]:
df["Dato Financiero"] = np.select(
    [
        df["PUC"].str.startswith("1"),
        df["PUC"].str.startswith("2"),
        df["PUC"].str.startswith("3"),
        df["PUC"].str.startswith("4"),
    ],
    [
        df["Saldo final"],
        -df["Saldo final"],
        -df["Saldo final"],
        df["Creditos "] - df["Debitos "],
    ],
    default=df["Debitos "] - df["Creditos "]
)


In [ ]:
df = df[[
    "Largo","Cuenta","PUC","Descripción","Tercero",
    "Saldo inicial","Debitos ","Creditos ","Saldo final","Dato Financiero"
]]

In [ ]:
df.head()

In [ ]:
# Exportar a excel
df.to_excel(
    r"C:\Users\User\Downloads\limpiado_CP.xlsx",  # Ruta de salida
    index=False                                      # No guardar índice
)